# Score Stability Experiments

这个 notebook 使用当前项目里的 `qwen3_ollama.py` + `src.scoring.pipeline` 评分链路，支持两个实验：

1. `data/experiments/same_pdf/` 里可以放多个 PDF；每个 PDF 默认跑 5 遍，统计每个文件自己的分数分布。
2. `data/experiments/group_a/` 和 `data/experiments/group_b/` 两组 PDF 做分布比较。

运行结果会自动写到 `data/experiments/results/`，解析后的 JSON 会缓存在 `data/experiments/results/parsed_cache/`。

默认复用解析缓存，只重复跑 scoring。需要连 parser 随机性一起测试时，把 `reparse_each_run=True`。


In [ ]:
!pip install matplotlib

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print('matplotlib is not installed; plotting helpers will be skipped.')
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
EXPERIMENT_ROOT = PROJECT_ROOT / 'data' / 'experiments'
SAME_PDF_DIR = EXPERIMENT_ROOT / 'same_pdf'
GROUP_A_DIR = EXPERIMENT_ROOT / 'group_a'
GROUP_B_DIR = EXPERIMENT_ROOT / 'group_b'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'

DEFAULT_SAME_PDF_RUNS_PER_FILE = 3
EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'  # 改这里即可切换实验模型，例如 'qwen3.5:35b'

for path in [SAME_PDF_DIR, GROUP_A_DIR, GROUP_B_DIR, RESULTS_DIR, PARSED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]
SECTION_SCORE_COLUMNS = [f'{section_key}_score_100' for section_key in SECTION_KEYS]
RUN_SCORE_COLUMNS = ['overall_score_100', *SECTION_SCORE_COLUMNS]
RUN_DISPLAY_COLUMNS = ['pdf_name', 'run_idx', *RUN_SCORE_COLUMNS, 'avg_signal_score_0to5']
GROUP_RUN_DISPLAY_COLUMNS = ['group', *RUN_DISPLAY_COLUMNS]

print('Project root:', PROJECT_ROOT)
print('Same PDF dir:', SAME_PDF_DIR)
print('Group A dir:', GROUP_A_DIR)
print('Group B dir:', GROUP_B_DIR)
print('Results dir:', RESULTS_DIR)
print('Default same-pdf runs per file:', DEFAULT_SAME_PDF_RUNS_PER_FILE)
print('Experiment Ollama model:', EXPERIMENT_OLLAMA_MODEL)


Project root: d:\MSc_AI\SWE_group_project\nlp_grant_coursework
Same PDF dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\same_pdf
Group A dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\group_a
Group B dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\group_b
Results dir: d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results
Default same-pdf runs per file: 3
Experiment Ollama model: qwen3.5:27b


In [2]:
def list_pdfs(folder: Path) -> list[Path]:
    return sorted([path for path in folder.glob('*.pdf') if path.is_file()])


def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    parsed = json.loads(json_path.read_text(encoding='utf-8'))
    return parsed, json_path


def make_experiment_scorer(model_name: str | None = None) -> _Scorer:
    return _Scorer(model_name=model_name or EXPERIMENT_OLLAMA_MODEL)


def score_pdf_once(
    pdf_path: Path,
    *,
    scorer: _Scorer,
    run_tag: str,
    reparse: bool = False,
) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / run_tag
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed,
        CRITERIA_PATH,
        doc_id=f'{pdf_path.stem}_{run_tag}',
        scorer=scorer,
        artifacts_dir=artifact_dir,
    )
    result['source_pdf'] = str(pdf_path)
    result['parsed_json'] = str(parsed_json_path)
    out_path = artifact_dir / f'{pdf_path.stem}_{run_tag}_scored.json'
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    result['result_json'] = str(out_path)
    return result


def average_signal_score(result: dict) -> float:
    scores = []
    for section in result.get('features', {}).values():
        for criterion in section.get('sub_criteria', section.get('criteria', [])):
            for signal in criterion.get('signals', []):
                scores.append(float(signal.get('score_0to5_raw', signal.get('score', 0)) or 0))
    return round(sum(scores) / len(scores), 4) if scores else 0.0


def flatten_result_row(result: dict, *, pdf_name: str, run_idx: int | None, group: str | None) -> dict:
    overall = result.get('overall', {})
    row = {
        'pdf_name': pdf_name,
        'run_idx': run_idx,
        'group': group,
        'overall_score_100': float(overall.get('final_score_0to100', 0)),
        'overall_score_10': float(overall.get('score_10', 0)),
        'quality_score_100': float(overall.get('quality_score_0to100', 0)),
        'coverage_score_100': float(overall.get('coverage_score_0to100', 0)),
        'avg_signal_score_0to5': average_signal_score(result),
        'total_items': int(overall.get('total_items', 0)),
        'signal_count': int(overall.get('signal_count', 0)),
        'good_items': int(overall.get('good_items', 0)),
        'positive_items': int(overall.get('positive_items', 0)),
        'pool_size': int(result.get('pool_size', 0)),
        'source_pdf': result.get('source_pdf'),
        'parsed_json': result.get('parsed_json'),
        'result_json': result.get('result_json'),
    }
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        section = features.get(section_key, {})
        section_overall = section.get('overall', {})
        row[f'{section_key}_score_100'] = float(section_overall.get('final_score_0to100', 0))
        row[f'{section_key}_score_10'] = float(section_overall.get('score_10', 0))
        row[f'{section_key}_quality_score_100'] = float(section_overall.get('quality_score_0to100', 0))
        row[f'{section_key}_coverage_score_100'] = float(section_overall.get('coverage_score_0to100', 0))
        row[f'{section_key}_signal_count'] = int(section_overall.get('signal_count', 0))
    return row


def score_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in RUN_SCORE_COLUMNS if col in df.columns]


def score_mean_std_columns(summary: pd.DataFrame) -> list[str]:
    id_cols = [col for col in ['pdf_name', 'group'] if col in summary.columns]
    metric_cols = []
    for metric in RUN_SCORE_COLUMNS:
        for suffix in ['mean', 'std']:
            col = f'{metric}_{suffix}'
            if col in summary.columns:
                metric_cols.append(col)
    if metric_cols:
        return id_cols + metric_cols
    return [col for col in ['metric', 'mean', 'std'] if col in summary.columns]


def format_run_scores(row: dict) -> str:
    parts = [f"overall={float(row.get('overall_score_100', 0)):.1f}"]
    for section_key in SECTION_KEYS:
        col = f'{section_key}_score_100'
        parts.append(f"{section_key}={float(row.get(col, 0)):.1f}")
    return ' | '.join(parts)


def summarize_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    summary = pd.DataFrame({
        'mean': df[columns].mean(),
        'std': df[columns].std(ddof=1),
        'var': df[columns].var(ddof=1),
        'min': df[columns].min(),
        'max': df[columns].max(),
        'median': df[columns].median(),
    }).reset_index().rename(columns={'index': 'metric'})
    return summary


def summarize_distribution(df: pd.DataFrame, *, by: str = 'pdf_name') -> pd.DataFrame:
    columns = score_columns(df)
    summary = df.groupby(by)[columns].agg(['count', 'mean', 'std', 'var', 'min', 'max', 'median'])
    summary.columns = ['_'.join(col).strip('_') for col in summary.columns.to_flat_index()]
    return summary.reset_index()


def plot_same_pdf(df: pd.DataFrame, pdf_name: str) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(df['run_idx'], df['overall_score_100'], marker='o')
    axes[0].set_title(f'Overall score by run: {pdf_name}')
    axes[0].set_xlabel('Run index')
    axes[0].set_ylabel('Score / 100')
    axes[0].grid(alpha=0.3)

    axes[1].boxplot(df['overall_score_100'], vert=True)
    axes[1].set_title('Overall score distribution')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def plot_same_pdf_distribution(df: pd.DataFrame) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    pdf_names = sorted(df['pdf_name'].unique())
    fig, axes = plt.subplots(1, 2, figsize=(max(14, len(pdf_names) * 2.2), 4.5))
    for pdf_name, pdf_df in df.groupby('pdf_name'):
        axes[0].plot(pdf_df['run_idx'], pdf_df['overall_score_100'], marker='o', label=pdf_name)
    axes[0].set_title('Overall score by run')
    axes[0].set_xlabel('Run index')
    axes[0].set_ylabel('Score / 100')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    grouped = [df.loc[df['pdf_name'] == pdf_name, 'overall_score_100'].tolist() for pdf_name in pdf_names]
    axes[1].boxplot(grouped, tick_labels=pdf_names)
    axes[1].set_title('Overall score distribution by PDF')
    axes[1].set_ylabel('Score / 100')
    axes[1].tick_params(axis='x', labelrotation=35)
    plt.tight_layout()
    plt.show()


def plot_group_compare(df: pd.DataFrame) -> None:
    if plt is None:
        print('matplotlib is not installed; skipping plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for group_name, group_df in df.groupby('group'):
        axes[0].hist(group_df['overall_score_100'], bins=min(10, max(3, len(group_df))), alpha=0.5, label=group_name)
    axes[0].set_title('Overall score distribution by group')
    axes[0].set_xlabel('Score / 100')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    grouped = [group_df['overall_score_100'].tolist() for _, group_df in sorted(df.groupby('group'))]
    labels = [group_name for group_name, _ in sorted(df.groupby('group'))]
    axes[1].boxplot(grouped, tick_labels=labels)
    axes[1].set_title('Overall score boxplot')
    axes[1].set_ylabel('Score / 100')
    plt.tight_layout()
    plt.show()


def optional_stat_tests(group_a_scores: pd.Series, group_b_scores: pd.Series) -> pd.DataFrame:
    rows = []
    try:
        from scipy.stats import ks_2samp, ttest_ind
        t_stat, t_p = ttest_ind(group_a_scores, group_b_scores, equal_var=False)
        ks_stat, ks_p = ks_2samp(group_a_scores, group_b_scores)
        rows.append({'test': 'welch_ttest', 'statistic': float(t_stat), 'p_value': float(t_p)})
        rows.append({'test': 'ks_2samp', 'statistic': float(ks_stat), 'p_value': float(ks_p)})
    except Exception as exc:
        rows.append({'test': 'scipy_unavailable', 'statistic': None, 'p_value': None, 'note': str(exc)})
    return pd.DataFrame(rows)


def run_same_pdf_experiment(
    pdf_path: Path,
    *,
    n_runs: int = DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run: bool = False,
    sleep_seconds: float = 0.0,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scorer = make_experiment_scorer(model_name)
    rows = []
    for run_idx in range(1, n_runs + 1):
        run_tag = f'same_pdf_{pdf_path.stem}_run_{run_idx:02d}'
        print(f'[{pdf_path.name}] run {run_idx}/{n_runs}')
        result = score_pdf_once(
            pdf_path,
            scorer=scorer,
            run_tag=run_tag,
            reparse=reparse_each_run,
        )
        row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group='same_pdf')
        rows.append(row)
        print(f'  scores | {format_run_scores(row)}')
        if sleep_seconds > 0:
            time.sleep(sleep_seconds)
    df = pd.DataFrame(rows)
    summary = summarize_numeric(df, score_columns(df))
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'same_pdf_{pdf_path.stem}_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'same_pdf_{pdf_path.stem}_summary_{timestamp}.csv', index=False)
    return df, summary


def run_same_pdf_distribution_experiment(
    pdf_paths: list[Path],
    *,
    runs_per_pdf: int = DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run: bool = False,
    sleep_seconds: float = 0.0,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    assert pdf_paths, f'No PDF files found in {SAME_PDF_DIR}'
    scorer = make_experiment_scorer(model_name)
    rows = []
    total_runs = len(pdf_paths) * runs_per_pdf
    completed = 0
    for pdf_path in pdf_paths:
        for run_idx in range(1, runs_per_pdf + 1):
            completed += 1
            run_tag = f'same_pdf_{pdf_path.stem}_run_{run_idx:02d}'
            print(f'[{completed}/{total_runs}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
            result = score_pdf_once(
                pdf_path,
                scorer=scorer,
                run_tag=run_tag,
                reparse=reparse_each_run,
            )
            row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group='same_pdf')
            rows.append(row)
            print(f'  scores | {format_run_scores(row)}')
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)
    df = pd.DataFrame(rows)
    summary = summarize_distribution(df, by='pdf_name')
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'same_pdf_distribution_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'same_pdf_distribution_summary_{timestamp}.csv', index=False)
    return df, summary


def run_group_compare_experiment(
    *,
    group_a_paths: list[Path],
    group_b_paths: list[Path],
    runs_per_pdf: int = 1,
    reparse_each_run: bool = False,
    model_name: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    scorer = make_experiment_scorer(model_name)
    rows = []
    for group_name, pdf_paths in [('group_a', group_a_paths), ('group_b', group_b_paths)]:
        for pdf_path in pdf_paths:
            for run_idx in range(1, runs_per_pdf + 1):
                run_tag = f'{group_name}_{pdf_path.stem}_run_{run_idx:02d}'
                print(f'[{group_name}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
                result = score_pdf_once(
                    pdf_path,
                    scorer=scorer,
                    run_tag=run_tag,
                    reparse=reparse_each_run,
                )
                row = flatten_result_row(result, pdf_name=pdf_path.name, run_idx=run_idx, group=group_name)
                rows.append(row)
                print(f'  scores | {format_run_scores(row)}')
    df = pd.DataFrame(rows)
    score_cols = score_columns(df)
    summary = df.groupby('group')[score_cols].agg(['count', 'mean', 'std', 'var', 'min', 'max', 'median'])
    summary.columns = ['_'.join(col).strip('_') for col in summary.columns.to_flat_index()]
    summary = summary.reset_index()
    tests = optional_stat_tests(
        df.loc[df['group'] == 'group_a', 'overall_score_100'],
        df.loc[df['group'] == 'group_b', 'overall_score_100'],
    )
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'group_compare_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'group_compare_summary_{timestamp}.csv')
    tests.to_csv(RESULTS_DIR / f'group_compare_tests_{timestamp}.csv', index=False)
    return df, summary, tests


## 实验 1：`same_pdf` 目录内多个 PDF，每个跑 5 遍

把一个或多个 PDF 放到 `data/experiments/same_pdf/`。下面这个 cell 会对目录里的每个 PDF 默认跑 5 遍，并输出每个 PDF 的分数分布。


In [ ]:
same_pdf_files = list_pdfs(SAME_PDF_DIR)
assert same_pdf_files, f'请在 {SAME_PDF_DIR} 里至少放 1 个 PDF。'

same_pdf_df, same_pdf_summary = run_same_pdf_distribution_experiment(
    same_pdf_files,
    runs_per_pdf=DEFAULT_SAME_PDF_RUNS_PER_FILE,
    reparse_each_run=False,
    sleep_seconds=0.0,
)

display(same_pdf_df[RUN_DISPLAY_COLUMNS])
display(same_pdf_summary[score_mean_std_columns(same_pdf_summary)])
plot_same_pdf_distribution(same_pdf_df)


[qwen3_ollama] using http://127.0.0.1:11434 model=qwen3.5:27b
[1/6] IC00001_DF_Doctoral.pdf run 1/3
[all_type_parser] ✓ fellowships_parser succeeded
[all_type_parser] saved → d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results\parsed_cache\IC00001_DF_Doctoral.json
  scores | overall=91.2 | general=93.0 | proposed_research=97.5 | training_development=86.7 | sites_support=95.0 | wpcc=90.0 | application_form=85.0
[2/6] IC00001_DF_Doctoral.pdf run 2/3
  scores | overall=88.1 | general=92.0 | proposed_research=94.6 | training_development=80.0 | sites_support=95.0 | wpcc=82.2 | application_form=85.0
[3/6] IC00001_DF_Doctoral.pdf run 3/3


## 实验 2：两组 PDF 分布比较

把两组 PDF 分别放到：

- `data/experiments/group_a/`
- `data/experiments/group_b/`

建议每组 5 个。默认每个 PDF 跑 1 次。

In [ ]:
group_a_files = list_pdfs(GROUP_A_DIR)
group_b_files = list_pdfs(GROUP_B_DIR)

assert len(group_a_files) == 5, f'请在 {GROUP_A_DIR} 里放 5 个 PDF，当前有 {len(group_a_files)} 个。'
assert len(group_b_files) == 5, f'请在 {GROUP_B_DIR} 里放 5 个 PDF，当前有 {len(group_b_files)} 个。'

group_df, group_summary, group_tests = run_group_compare_experiment(
    group_a_paths=group_a_files,
    group_b_paths=group_b_files,
    runs_per_pdf=1,
    reparse_each_run=False,
)

display(group_df[GROUP_RUN_DISPLAY_COLUMNS])
display(group_summary[score_mean_std_columns(group_summary)])
display(group_tests)
plot_group_compare(group_df)


## 实验 3：Baseline —— 整个 parsed JSON 直接单次调用 LLM 打分（signal 级别）

**目的**：与实验 1（pipeline 多阶段打分）对比，评估两阶段流水线相对于最朴素 baseline 的增益。

**方法**：
- 不做任何预处理（无 chunking、无 Stage 1 证据检索、无分 section 调用）
- 把完整 parsed JSON + `criteria_points.json` 的所有 sub-criteria / signal 文本一次性喂给 LLM
- LLM 对**每个 signal** 输出 0–5 的整数分数
- 聚合逻辑与 pipeline 完全一致：signal → sub-criterion（0–10）→ section（0–100）→ overall（0–100）
- 与实验 1 在 overall、section、signal 三个层级做并排比较

In [12]:
# ── 实验 3 helpers ──────────────────────────────────────────────────────────
from src.scoring.pipeline import (
    load_rubric,
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
)

BASELINE_NUM_CTX = 49152   # Ollama context window (tokens). Increase if scores are all 0.


class _BaselineScorer:
    """Thin wrapper that injects num_ctx into every Ollama request."""

    def __init__(self, model_name: str = EXPERIMENT_OLLAMA_MODEL, num_ctx: int = BASELINE_NUM_CTX):
        from qwen3_ollama import OLLAMA_HOST, OLLAMA_TIMEOUT
        import requests as _requests
        self.model_name = model_name
        self.num_ctx = num_ctx
        self._host = OLLAMA_HOST
        self._timeout = OLLAMA_TIMEOUT
        self._requests = _requests
        self.last_response_body = None
        print(f'[baseline] model={model_name}  num_ctx={num_ctx}', flush=True)

    def generate_json(self, messages: list, *, schema: dict, max_tokens: int) -> str:
        import re as _re
        payload = {
            'model': self.model_name,
            'messages': messages,
            'stream': False,
            'format': schema,
            'options': {
                'temperature': 0.1,
                'top_p': 0.9,
                'num_predict': max_tokens,
                'num_ctx': self.num_ctx,
            },
            'think': False,
        }
        try:
            resp = self._requests.post(
                f'{self._host}/api/chat', json=payload, timeout=self._timeout,
            )
        except self._requests.exceptions.ConnectionError as exc:
            raise RuntimeError(f'Could not connect to Ollama at {self._host}.') from exc
        except self._requests.exceptions.Timeout as exc:
            raise RuntimeError('Timed out waiting for Ollama.') from exc
        resp.raise_for_status()
        body = resp.json()
        self.last_response_body = body
        message = body.get('message') or {}
        content = _re.sub(
            r'<think>.*?</think>\s*', '', message.get('content') or '', flags=_re.DOTALL,
        ).strip()
        first, last = content.find('{'), content.rfind('}')
        return content[first:last + 1] if first != -1 and last > first else content


BASELINE_SECTION_KEYS = [
    'general', 'proposed_research', 'training_development',
    'sites_support', 'wpcc', 'application_form',
]


# ── Schema: sub-criterion level only (0-10 float) ────────────────────────────
def _build_baseline_sub_schema(rubric_sections: list) -> dict:
    """
    Schema constrains only sub-criterion IDs → number 0-10.
    Signals are reference only — no signal-level enforcement.
    This makes the baseline LESS constrained than the pipeline.
    """
    properties = {
        sub['sub_id']: {'type': 'number', 'minimum': 0, 'maximum': 10}
        for sec in rubric_sections
        for sub in sec['sub_criteria']
    }
    return {
        'type': 'object',
        'properties': properties,
        'required': list(properties.keys()),
        'additionalProperties': False,
    }


# ── Prompt: image-style, subcriteria 0-10, signals as reference ──────────────
def _format_criteria_for_prompt(rubric_sections: list) -> str:
    """Format all sub-criteria and their signals as readable reference text."""
    lines = []
    for section in rubric_sections:
        lines.append(f"\n=== {section['human_name']} ===")
        for sub in section['sub_criteria']:
            lines.append(f"  [{sub['sub_id']}] {sub['name']}")
            lines.append(f"  Definition: {sub['definition']}")
            for sig in sub['signals']:
                lines.append(f"    - {sig['text']}")
    return '\n'.join(lines)


def _build_baseline_messages(parsed_app: dict, rubric_sections: list) -> list:
    """
    Image-style prompt: score each sub-criterion 0-10, use signals as reference.
    Less constrained than pipeline: no section-by-section decomposition,
    no evidence retrieval, single call for the entire application.
    """
    criteria_text = _format_criteria_for_prompt(rubric_sections)
    all_sub_ids = [
        sub['sub_id']
        for sec in rubric_sections
        for sub in sec['sub_criteria']
    ]
    if_applicable_subs = [
        sub['sub_id']
        for sec in rubric_sections
        for sub in sec['sub_criteria']
        if 'if applicable' in f"{sub.get('name','')} {sub.get('definition','')}".lower()
    ]
    system = (
        'You are an expert grant application reviewer.\n'
        'Score the grant application according to the scoring rubric below.\n'
        'For each sub-criterion, assign a score from 0 to 10.\n'
        'Use the signals listed under each sub-criterion as reference '
        'when deciding the score — do NOT score each signal individually.\n'
        'Sub-criteria marked as "if applicable" that do not apply to this '
        'application should receive a score of -1 to indicate N/A.\n'
        'Return JSON only.\n'
        'Structure: top-level keys are sub-criterion IDs, values are numbers (0-10, or -1 for N/A).\n'
        f'Required keys: {all_sub_ids}\n'
        f'Potentially N/A sub-criteria: {if_applicable_subs}'
    )
    user = (
        'SCORING RUBRIC (signals are reference only — score at sub-criterion level):\n'
        + criteria_text
        + '\n\nGRANT APPLICATION:\n'
        + json.dumps(parsed_app, ensure_ascii=False, indent=2)
        + '\n\nReturn a JSON object mapping each sub-criterion ID to its 0-10 score '
        '(or -1 if not applicable).'
    )
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user},
    ]


# ── Aggregation: sub-level input, same section/overall math as pipeline ───────
def _aggregate_baseline(raw_scores: dict, rubric_sections: list, doc_type: str = '') -> tuple:
    """
    raw_scores: {sub_id: float(0-10)}  (-1 means N/A, excluded from average)
    Aggregation mirrors pipeline _aggregate_section / _aggregate_overall:
      sub score_10 (weighted) → section score_10 → overall score_10
    """
    dt = doc_type.lower()
    excluded_sections = OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE.get(dt, set())
    excluded_sub_ids  = SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE.get(dt, set())

    features = {}
    all_sub_scores = []

    for section in rubric_sections:
        section_key = section['section_key']
        sub_results = []

        for sub in section['sub_criteria']:
            sub_id = sub['sub_id']
            raw_val = raw_scores.get(sub_id, 0)
            # -1 = N/A (LLM flagged as not applicable)
            is_na = isinstance(raw_val, (int, float)) and raw_val < 0
            score_10 = 0.0 if is_na else round(max(0.0, min(10.0, float(raw_val))), 2)
            is_doc_excluded = sub_id in excluded_sub_ids
            counts = not is_na and not is_doc_excluded
            sub_results.append({
                'sub_id': sub_id,
                'name': sub['name'],
                'weight': sub['weight'],
                'score_10': score_10,
                'is_na': is_na,
                'counts_toward_section_average': counts,
            })
            if counts:
                all_sub_scores.append(score_10)

        # section score: same formula as pipeline _aggregate_section
        scored = [s for s in sub_results if s['counts_toward_section_average']] or sub_results
        total_w = sum(s['weight'] for s in scored) or 1.0
        section_score_10 = round(
            sum(s['score_10'] * s['weight'] for s in scored) / total_w, 2
        )
        positive_items = sum(1 for s in scored if s['score_10'] > 0)

        features[section_key] = {
            'score_10': section_score_10,
            'weight': section['weight'],
            'sub_criteria': sub_results,
            'overall': {
                'score_10': section_score_10,
                'final_score_0to100': round(section_score_10 * 10, 2),
                'positive_items': positive_items,
                'scored_items': len(scored),
                'total_items': len(sub_results),
                'signal_count': sum(len(sub['signals']) for sub in section['sub_criteria']),
            },
        }

    # overall: same formula as pipeline _aggregate_overall
    scoring_features = {
        k: v for k, v in features.items() if k not in excluded_sections
    } or features
    section_weights = {k: v['weight'] for k, v in features.items()}
    total_w = sum(section_weights.get(k, 1.0) for k in scoring_features) or 1.0
    overall_10 = round(
        sum(features[k]['score_10'] * section_weights.get(k, 1.0) for k in scoring_features) / total_w, 2
    )
    overall = {
        'score_10': overall_10,
        'final_score_0to100': round(overall_10 * 10, 2),
    }
    # avg_sub_score on 0-5 scale for comparison with pipeline's avg_signal_score_0to5
    avg_sub_as_signal = round(sum(all_sub_scores) / len(all_sub_scores) / 2, 4) if all_sub_scores else 0.0
    return features, overall, avg_sub_as_signal


# ── Scoring entry point ───────────────────────────────────────────────────────
def baseline_score_pdf_once(
    pdf_path: Path,
    *,
    scorer,
    run_tag: str,
    reparse: bool = False,
    criteria_path: Path = CRITERIA_PATH,
) -> dict:
    """parse（带缓存）→ 单次 LLM 调用（sub-criterion 级分数）→ pipeline 相同的聚合逻辑。"""
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    rubric_sections = load_rubric(criteria_path)
    doc_type = (parsed.get('doc_type') or '').lower()

    messages = _build_baseline_messages(parsed, rubric_sections)
    schema   = _build_baseline_sub_schema(rubric_sections)
    raw = scorer.generate_json(messages, schema=schema, max_tokens=1024)

    try:
        raw_scores = json.loads(raw)
    except json.JSONDecodeError:
        import re as _re
        m = _re.search(r'\{.*\}', raw, _re.DOTALL)
        raw_scores = json.loads(m.group()) if m else {}

    features, overall, avg_sub = _aggregate_baseline(raw_scores, rubric_sections, doc_type)

    artifact_dir = RESULTS_DIR / run_tag
    artifact_dir.mkdir(parents=True, exist_ok=True)
    out_path = artifact_dir / f'{pdf_path.stem}_{run_tag}_baseline.json'
    result = {
        'doc_id': f'{pdf_path.stem}_{run_tag}',
        'source_pdf': str(pdf_path),
        'parsed_json': str(parsed_json_path),
        'features': features,
        'overall': overall,
        'avg_signal_score_0to5': avg_sub,  # avg sub score converted to 0-5 for comparison
        'raw_sub_scores': raw_scores,
        'raw_llm_response': raw,
    }
    out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    result['result_json'] = str(out_path)
    return result


def flatten_baseline_row(result: dict, *, pdf_name: str, run_idx: int) -> dict:
    overall  = result.get('overall', {})
    features = result.get('features', {})
    row = {
        'pdf_name': pdf_name,
        'run_idx': run_idx,
        'overall_score_100': float(overall.get('final_score_0to100', 0)),
        'avg_signal_score_0to5': float(result.get('avg_signal_score_0to5', 0)),
        'source_pdf': result.get('source_pdf'),
        'result_json': result.get('result_json'),
    }
    for key in BASELINE_SECTION_KEYS:
        sec = features.get(key, {})
        row[f'{key}_score_100'] = float(sec.get('overall', {}).get('final_score_0to100', 0))
    return row


def run_baseline_experiment(
    pdf_paths: list,
    *,
    runs_per_pdf: int = 3,
    reparse_each_run: bool = False,
    model_name: str | None = None,
) -> tuple:
    """对 same_pdf 目录里的 PDF 跑 baseline，每个 PDF 跑 runs_per_pdf 遍。"""
    assert pdf_paths, 'No PDF files found.'
    scorer = _BaselineScorer(model_name=model_name or EXPERIMENT_OLLAMA_MODEL)
    rows = []
    total = len(pdf_paths) * runs_per_pdf
    completed = 0
    for pdf_path in pdf_paths:
        for run_idx in range(1, runs_per_pdf + 1):
            completed += 1
            run_tag = f'baseline_{pdf_path.stem}_run_{run_idx:02d}'
            print(f'[baseline {completed}/{total}] {pdf_path.name} run {run_idx}/{runs_per_pdf}')
            result = baseline_score_pdf_once(
                pdf_path, scorer=scorer, run_tag=run_tag, reparse=reparse_each_run,
            )
            row = flatten_baseline_row(result, pdf_name=pdf_path.name, run_idx=run_idx)
            rows.append(row)
            section_parts = ' | '.join(
                f'{k}={row[f"{k}_score_100"]:.1f}' for k in BASELINE_SECTION_KEYS
            )
            print(f"  overall={row['overall_score_100']:.1f} | avg_sub(0-5)={row['avg_signal_score_0to5']:.2f}")
            print(f'  sections: {section_parts}')

    df = pd.DataFrame(rows)
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    summary = summarize_numeric(df, score_cols)
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    df.to_csv(RESULTS_DIR / f'baseline_runs_{timestamp}.csv', index=False)
    summary.to_csv(RESULTS_DIR / f'baseline_summary_{timestamp}.csv', index=False)
    return df, summary


def compare_pipeline_vs_baseline(pipeline_df: pd.DataFrame, baseline_df: pd.DataFrame) -> pd.DataFrame:
    """并排比较 pipeline（实验1）和 baseline（实验3）在同一组 PDF 上的均值分数。"""
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    p_rename = {c: f'pipeline_{c}' for c in score_cols}
    b_rename = {c: f'baseline_{c}' for c in score_cols}
    p_mean = pipeline_df.groupby('pdf_name')[
        [c for c in score_cols if c in pipeline_df.columns]
    ].mean().rename(columns=p_rename)
    b_mean = baseline_df.groupby('pdf_name')[
        [c for c in score_cols if c in baseline_df.columns]
    ].mean().rename(columns=b_rename)
    merged = p_mean.join(b_mean, how='outer')
    for c in score_cols:
        pc, bc = f'pipeline_{c}', f'baseline_{c}'
        if pc in merged.columns and bc in merged.columns:
            merged[f'diff_{c}'] = merged[pc] - merged[bc]
    return merged.reset_index()


def summarize_per_pdf(df: pd.DataFrame) -> pd.DataFrame:
    """对每个 PDF 分别统计多次 baseline 运行的 mean / std / min / max。"""
    score_cols = ['overall_score_100', 'avg_signal_score_0to5'] + [
        f'{k}_score_100' for k in BASELINE_SECTION_KEYS
    ]
    available = [c for c in score_cols if c in df.columns]
    agg = df.groupby('pdf_name')[available].agg(['mean', 'std', 'min', 'max'])
    agg.columns = ['_'.join(col) for col in agg.columns]
    return agg.reset_index()


In [13]:
# ── 实验 3 运行 ──────────────────────────────────────────────────────────────
# 与实验 1 使用相同模型和相同跑数，确保对比公平。
# 复用实验 1 的 parsed 缓存，仅重新调用 LLM。
# num_ctx 由 BASELINE_NUM_CTX 控制（默认 32768），可在 cell 9 顶部修改。
BASELINE_MODEL        = EXPERIMENT_OLLAMA_MODEL
BASELINE_RUNS_PER_PDF = DEFAULT_SAME_PDF_RUNS_PER_FILE

same_pdf_files = list_pdfs(SAME_PDF_DIR)
assert same_pdf_files, f'请在 {SAME_PDF_DIR} 里至少放 1 个 PDF。'

baseline_df, _ = run_baseline_experiment(
    same_pdf_files,
    runs_per_pdf=BASELINE_RUNS_PER_PDF,
    reparse_each_run=False,
    model_name=BASELINE_MODEL,
)

BASELINE_DISPLAY_COLS = [
    'pdf_name', 'run_idx', 'overall_score_100', 'avg_signal_score_0to5',
] + [f'{k}_score_100' for k in BASELINE_SECTION_KEYS]

print('\n── Baseline 各次运行分数 ──')
display(baseline_df[BASELINE_DISPLAY_COLS])

print('\n── Baseline 每个 PDF 的跨运行统计（mean / std / min / max）──')
baseline_per_pdf_summary = summarize_per_pdf(baseline_df)
display(baseline_per_pdf_summary)


[baseline] model=qwen3.5:27b  num_ctx=49152
[baseline 1/6] IC00001_DF_Doctoral.pdf run 1/3
  overall=89.8 | avg_sub(0-5)=4.45
  sections: general=88.3 | proposed_research=87.8 | training_development=90.0 | sites_support=97.5 | wpcc=85.0 | application_form=90.0
[baseline 2/6] IC00001_DF_Doctoral.pdf run 2/3
  overall=90.9 | avg_sub(0-5)=4.48
  sections: general=88.3 | proposed_research=87.8 | training_development=96.7 | sites_support=97.5 | wpcc=85.0 | application_form=90.0
[baseline 3/6] IC00001_DF_Doctoral.pdf run 3/3
  overall=88.5 | avg_sub(0-5)=4.40
  sections: general=88.3 | proposed_research=86.7 | training_development=90.0 | sites_support=97.5 | wpcc=85.0 | application_form=83.3
[baseline 4/6] IC00147_after.pdf run 1/3
  overall=89.2 | avg_sub(0-5)=4.45
  sections: general=90.0 | proposed_research=87.8 | training_development=90.0 | sites_support=97.5 | wpcc=86.7 | application_form=83.3
[baseline 5/6] IC00147_after.pdf run 2/3
  overall=89.2 | avg_sub(0-5)=4.45
  sections: genera

,pdf_name,run_idx,overall_score_100,avg_signal_score_0to5,general_score_100,proposed_research_score_100,training_development_score_100,sites_support_score_100,wpcc_score_100,application_form_score_100
0,IC00001_DF_Doctoral.pdf,1,89.8,4.4516,88.3,87.8,90.0,97.5,85.0,90.0
1,IC00001_DF_Doctoral.pdf,2,90.9,4.4839,88.3,87.8,96.7,97.5,85.0,90.0
2,IC00001_DF_Doctoral.pdf,3,88.5,4.4032,88.3,86.7,90.0,97.5,85.0,83.3
3,IC00147_after.pdf,1,89.2,4.4516,90.0,87.8,90.0,97.5,86.7,83.3
4,IC00147_after.pdf,2,89.2,4.4516,90.0,87.8,90.0,97.5,86.7,83.3
5,IC00147_after.pdf,3,91.2,4.5484,90.0,90.0,96.7,97.5,90.0,83.3



── Baseline 每个 PDF 的跨运行统计（mean / std / min / max）──


,pdf_name,overall_score_100_mean,overall_score_100_std,overall_score_100_min,overall_score_100_max,avg_signal_score_0to5_mean,avg_signal_score_0to5_std,avg_signal_score_0to5_min,avg_signal_score_0to5_max,general_score_100_mean,...,sites_support_score_100_min,sites_support_score_100_max,wpcc_score_100_mean,wpcc_score_100_std,wpcc_score_100_min,wpcc_score_100_max,application_form_score_100_mean,application_form_score_100_std,application_form_score_100_min,application_form_score_100_max
0,IC00001_DF_Doctoral.pdf,89.733333,1.201388,88.5,90.9,4.446233,0.040617,4.4032,4.4839,88.3,...,97.5,97.5,85.0,0.000000,85.0,85.0,87.766667,3.868247,83.3,90.0
1,IC00147_after.pdf,89.866667,1.154701,89.2,91.2,4.483867,0.055888,4.4516,4.5484,90.0,...,97.5,97.5,87.8,1.905256,86.7,90.0,83.300000,0.000000,83.3,83.3


In [ ]:
# ── 实验 1 vs 实验 3 对比 ────────────────────────────────────────────────────
# 依赖：same_pdf_df（实验1）和 baseline_df（实验3）都已在内存中。

try:
    comparison_df = compare_pipeline_vs_baseline(same_pdf_df, baseline_df)
    print('── Pipeline（实验1）vs Baseline（实验3）均值对比 ──')
    display(comparison_df)

    if plt is not None:
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))

        # ── 图1：overall 分数对比（每个 PDF 一组柱）──────────────────────────────
        pdf_names = comparison_df['pdf_name'].tolist()
        x = list(range(len(pdf_names)))
        w = 0.35
        axes[0].bar(
            [i - w/2 for i in x], comparison_df['pipeline_overall_score_100'],
            w, label='Pipeline（实验1）', color='steelblue',
        )
        axes[0].bar(
            [i + w/2 for i in x], comparison_df['baseline_overall_score_100'],
            w, label='Baseline（实验3）', color='darkorange',
        )
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(pdf_names, rotation=30, ha='right', fontsize=8)
        axes[0].set_ylabel('Overall Score / 100')
        axes[0].set_title('Overall Score: Pipeline vs Baseline')
        axes[0].legend()
        axes[0].grid(axis='y', alpha=0.3)

        # ── 图2：各 section 分数差（pipeline - baseline）均值 ─────────────────────
        section_diffs = []
        for key in BASELINE_SECTION_KEYS:
            col = f'diff_{key}_score_100'
            if col in comparison_df.columns:
                section_diffs.append((key, comparison_df[col].mean()))
        if section_diffs:
            labels, vals = zip(*section_diffs)
            colors = ['steelblue' if v >= 0 else 'salmon' for v in vals]
            axes[1].bar(range(len(labels)), vals, color=colors)
            axes[1].axhline(0, color='black', linewidth=0.8)
            axes[1].set_xticks(range(len(labels)))
            axes[1].set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
            axes[1].set_ylabel('Pipeline − Baseline (mean)')
            axes[1].set_title('Section-level Difference\n(+: pipeline higher)')
            axes[1].grid(axis='y', alpha=0.3)

        # ── 图3：avg_signal_score_0to5 对比 ──────────────────────────────────────
        p_sig_col = 'pipeline_avg_signal_score_0to5'
        b_sig_col = 'baseline_avg_signal_score_0to5'
        if p_sig_col in comparison_df.columns and b_sig_col in comparison_df.columns:
            axes[2].bar(
                [i - w/2 for i in x], comparison_df[p_sig_col],
                w, label='Pipeline（实验1）', color='steelblue',
            )
            axes[2].bar(
                [i + w/2 for i in x], comparison_df[b_sig_col],
                w, label='Baseline（实验3）', color='darkorange',
            )
            axes[2].set_xticks(x)
            axes[2].set_xticklabels(pdf_names, rotation=30, ha='right', fontsize=8)
            axes[2].set_ylabel('Avg Signal Score (0–5)')
            axes[2].set_title('Avg Signal Score: Pipeline vs Baseline')
            axes[2].legend()
            axes[2].grid(axis='y', alpha=0.3)
        else:
            axes[2].set_visible(False)

        plt.tight_layout()
        plt.show()

except NameError:
    print('same_pdf_df 未找到，请先运行实验 1 的 cell 以生成 same_pdf_df，再运行此对比 cell。')


## 实验 3 诊断：IC00147 分数全 0 排查

排查步骤：
1. **从已保存的 artifact 里读取原始 LLM 输出** — 看 raw_llm_response 是什么
2. **估算输入 token 数** — 判断是否超出 num_ctx
3. **重新跑单次调用** — 实时抓取 raw 输出及 Ollama 返回的 done_reason

In [9]:
# ── 诊断 1：读取已保存 artifact，查看 raw_llm_response ─────────────────────────
import glob as _glob

TARGET_PDF_STEM = 'IC00147_after'   # 修改此处可排查其他 PDF

# 找所有该 PDF 的 baseline artifact
pattern = str(RESULTS_DIR / f'**/*{TARGET_PDF_STEM}*_baseline.json')
artifact_files = sorted(_glob.glob(pattern, recursive=True))
print(f'找到 {len(artifact_files)} 个 artifact:')
for p in artifact_files:
    print(' ', p)

if artifact_files:
    latest = artifact_files[-1]
    art = json.loads(Path(latest).read_text(encoding='utf-8'))
    raw_resp = art.get('raw_llm_response', '') or art.get('raw_signal_scores', {})
    raw_sub  = art.get('raw_sub_scores', art.get('raw_signal_scores', {}))

    print(f'\n── raw_llm_response (前 2000 字符) ──')
    print(repr(str(raw_resp)[:2000]))

    print(f'\n── raw_sub_scores（解析后，共 {len(raw_sub)} 个 sub） ──')
    print(json.dumps(raw_sub, indent=2, ensure_ascii=False)[:1000])

    # 检查是否有非零分
    non_zero = {k: v for k, v in raw_sub.items() if isinstance(v, (int, float)) and v != 0}
    print(f'\n非零分 sub-criteria 数：{len(non_zero)} / {len(raw_sub)}')
    if non_zero:
        print(json.dumps(non_zero, indent=2))
    else:
        print('全部为 0 或空 — LLM 输出可能被截断或格式化失败')
else:
    print('未找到 artifact，请先运行 cell 10（实验 3）。')


找到 3 个 artifact:
  d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results\baseline_IC00147_after_run_01\IC00147_after_baseline_IC00147_after_run_01_baseline.json
  d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results\baseline_IC00147_after_run_02\IC00147_after_baseline_IC00147_after_run_02_baseline.json
  d:\MSc_AI\SWE_group_project\nlp_grant_coursework\data\experiments\results\baseline_IC00147_after_run_03\IC00147_after_baseline_IC00147_after_run_03_baseline.json

── raw_llm_response (前 2000 字符) ──
'{\n  "PPIE_1": 9,\n  "PPIE_2": 9,\n  "PPIE_3": -1,\n  "TRAINING_1": 10,\n  "TRAINING_2": 10,\n  "TRAINING_3": 10,\n  "BUDGET_1": 10,\n  "BUDGET_2": 10,\n  "BUDGET_3": 10,\n  "BUDGET_4": 10,\n  "BUDGET_5": 10\n}'

── raw_sub_scores（解析后，共 11 个 sub） ──
{
  "PPIE_1": 9,
  "PPIE_2": 9,
  "PPIE_3": -1,
  "TRAINING_1": 10,
  "TRAINING_2": 10,
  "TRAINING_3": 10,
  "BUDGET_1": 10,
  "BUDGET_2": 10,
  "BUDGET_3": 10,
  "BUDGET_4": 10,
  "BUDGET_5": 10
}

非零分 

In [10]:
# ── 诊断 2：估算输入 token 数，判断是否超出 num_ctx ────────────────────────────
# 用字符数 / 3.5 粗估 token 数（英文约 4 chars/token，中文约 2 chars/token）

TARGET_PDF_STEM2 = 'IC00147_after'   # 可修改

# 找对应的 parsed JSON
parsed_cache_files = sorted(
    (PARSED_CACHE_DIR / f'{TARGET_PDF_STEM2}.json').parent.glob(f'{TARGET_PDF_STEM2}*.json')
)
if not parsed_cache_files:
    print('未找到 parsed cache，请先运行一次 parse_pdf_cached 或实验 3。')
else:
    parsed_path = parsed_cache_files[0]
    parsed_app = json.loads(parsed_path.read_text(encoding='utf-8'))
    rubric_sections_diag = load_rubric(CRITERIA_PATH)
    msgs = _build_baseline_messages(parsed_app, rubric_sections_diag)

    system_chars = len(msgs[0]['content'])
    user_chars   = len(msgs[1]['content'])
    total_chars  = system_chars + user_chars
    est_tokens   = total_chars / 3.5

    print(f'system message : {system_chars:>8,} chars')
    print(f'user message   : {user_chars:>8,} chars')
    print(f'total          : {total_chars:>8,} chars')
    print(f'estimated tokens (÷3.5) : {est_tokens:>8,.0f}')
    print(f'BASELINE_NUM_CTX        : {BASELINE_NUM_CTX:>8,}')
    if est_tokens > BASELINE_NUM_CTX * 0.9:
        print('⚠️  INPUT LIKELY EXCEEDS num_ctx — increase BASELINE_NUM_CTX in cell 9!')
    else:
        print('Input appears within context window.')

    # Show parsed JSON size breakdown
    app_json_chars = len(json.dumps(parsed_app, ensure_ascii=False))
    criteria_chars = total_chars - app_json_chars
    print(f'\nBreakdown: application JSON = {app_json_chars:,} chars | criteria+prompt = {criteria_chars:,} chars')


system message :      827 chars
user message   :  148,882 chars
total          :  149,709 chars
estimated tokens (÷3.5) :   42,774
BASELINE_NUM_CTX        :   32,768
⚠️  INPUT LIKELY EXCEEDS num_ctx — increase BASELINE_NUM_CTX in cell 9!

Breakdown: application JSON = 135,238 chars | criteria+prompt = 14,471 chars


In [11]:
# ── 诊断 3：对 IC00147 重新跑单次调用，实时显示 done_reason + raw 输出 ─────────

TARGET_PDF_STEM3 = 'IC00147_after'   # 可修改

target_pdf_path = None
for p in list_pdfs(SAME_PDF_DIR):
    if TARGET_PDF_STEM3 in p.stem:
        target_pdf_path = p
        break

if target_pdf_path is None:
    print(f'{TARGET_PDF_STEM3} 不在 {SAME_PDF_DIR}，请把对应 PDF 放进去。')
else:
    diag_scorer = _BaselineScorer(model_name=EXPERIMENT_OLLAMA_MODEL)
    parsed_diag, _ = parse_pdf_cached(target_pdf_path)
    rubric_diag    = load_rubric(CRITERIA_PATH)
    messages_diag  = _build_baseline_messages(parsed_diag, rubric_diag)
    schema_diag    = _build_baseline_sub_schema(rubric_diag)

    print('Sending request to Ollama...')
    raw_diag = diag_scorer.generate_json(messages_diag, schema=schema_diag, max_tokens=1024)

    # done_reason tells us if output was cut off
    body = diag_scorer.last_response_body or {}
    done_reason   = body.get('done_reason', 'unknown')
    prompt_tokens = body.get('prompt_eval_count', 'n/a')
    gen_tokens    = body.get('eval_count', 'n/a')
    print(f'\ndone_reason    : {done_reason}')
    print(f'prompt_tokens  : {prompt_tokens}  (actual tokens sent)')
    print(f'generated tokens: {gen_tokens}')
    if done_reason == 'length':
        print('⚠️  Output was CUT OFF (done_reason=length). Increase BASELINE_NUM_CTX or max_tokens.')
    elif done_reason == 'stop':
        print('Output completed normally.')

    print(f'\n── raw LLM output ({len(raw_diag)} chars) ──')
    print(raw_diag[:3000])

    # Try parsing
    try:
        parsed_diag_scores = json.loads(raw_diag)
        non_zero_diag = {k: v for k, v in parsed_diag_scores.items() if v and v != 0}
        print(f'\nParsed OK — {len(parsed_diag_scores)} sub-criteria, {len(non_zero_diag)} non-zero')
        print(json.dumps(parsed_diag_scores, indent=2, ensure_ascii=False)[:1500])
    except json.JSONDecodeError as e:
        print(f'\nJSON parse FAILED: {e}')
        print('Raw output is not valid JSON — likely truncated or schema enforcement failed.')


[baseline] model=qwen3.5:27b  num_ctx=32768
Sending request to Ollama...

done_reason    : stop
prompt_tokens  : 32768  (actual tokens sent)
generated tokens: 78
Output completed normally.

── raw LLM output (223 chars) ──
{
  "patient_involvement_development": 9,
  "patient_involvement_implementation": 9,
  "patient_involvement_justification": -1,
  "training_and_development": 9,
  "supervision_and_support": 10,
  "budget_justification": 9
}

Parsed OK — 6 sub-criteria, 6 non-zero
{
  "patient_involvement_development": 9,
  "patient_involvement_implementation": 9,
  "patient_involvement_justification": -1,
  "training_and_development": 9,
  "supervision_and_support": 10,
  "budget_justification": 9
}
